In [0]:
spark

In [0]:
# display(dbutils.fs.ls("/databricks-datasets/airlines/"))

In [0]:
# %fs ls /databricks-datasets

In [0]:
# dbutils.fs.ls("dbfs:/databricks-datasets/airlines")


## finding file type 

In [0]:
%pip install python-magic
import magic

path = "/dbfs/databricks-datasets/airlines/part-00000"

file_type = magic.from_file(path, mime=True)
print(f"Detected MIME type: {file_type}")


### Reading of .md file 

In [0]:
md_path = "dbfs:/databricks-datasets/airlines/README.md"
content = dbutils.fs.head(md_path)  # reads first 10,000 bytes
print(content)

In [0]:
# df = spark.read.format("csv").option("inferschema",True).load("dbfs:/databricks-datasets/airlines")

In [0]:
# display(df)

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/sample/json/"))

In [0]:
amazon_df = spark.read.format("json").option("inferschema",True).load("dbfs:/databricks-datasets/nyctaxi/sample/json/")
display(amazon_df)


In [0]:
data = [1, 2, 3, 4, 5, 6]
rdd = spark.sparkContext.parallelize(data)
rdd.collect()



## Rdd 

In [0]:
from pyspark.sql import SparkSession
data = ["apple","banana","grape"]
rdd = spark.sparkContext.parallelize(data)
upper_rdd = rdd.map(lambda x: x.upper())
print(upper_rdd.collect())

In [0]:
from pyspark.sql import SparkSession

data = ["apple", "banana", "grape"]
df = spark.createDataFrame(data, "string").toDF("fruit")
upper_df = df.selectExpr("upper(fruit) as fruit_upper")
display(upper_df)

In [0]:
# Coded in PySpark

# Let's assume we already have a DataFrame with an integer column 'price'
# For example:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("LazyEvalExample").getOrCreate()
data = [(10,), (20,), (30,)]
df = spark.createDataFrame(data, ["price"])

# Transformation 1: Multiply price by 2
df = df.withColumn('price', col('price') * 2)

# Transformation 2: Multiply price by 3
df = df.withColumn('price', col('price') * 3)

# Transformation 3: Multiply price by 5
df = df.withColumn('price', col('price') * 5)

# No execution has happened yet! (Lazy Evaluation)

# This action triggers the execution
df_collect = df.collect()

# Output: [Row(price=300), Row(price=600), Row(price=900)]
print(df_collect)


# b35&b36 coding starts

### csv files path: https://github.com/datablist/sample-csv-files?tab=readme-ov-file


##  Create DataFrame from Dictionary: 

In [0]:
data_dict = [{"ID": 1, "Name": "Alice"}, {"ID": 2, "Name": "Bob"}] 
df_from_dict = spark.createDataFrame(data_dict) 
df_from_dict.show() 

##  Create Empty DataFrame

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
schema = StructType([ 
                     StructField("id", IntegerType(), True),
                     StructField("name", StringType(), True)
                     ])

In [0]:
empty_df = spark.createDataFrame([],schema)
display(empty_df)

## 5. Creating DataFrame from Structured Data (CSV, JSON, Parquet

In [0]:
df_csv = spark.read.csv("/Volumes/manu/default/csv_file", header=True, 
inferSchema=True) 
df_csv.display()

In [0]:
# df_csv.printSchema()

###Show the first 3 rows, truncate columns to 25 characters, and display vertically: 

In [0]:
df_csv.show(n=3, truncate=25, vertical=True)

###Show the first 10 rows:

In [0]:
df_csv.show(10) 

## Loading Data from CSV File into a DataFrame

1.Import Required Libraries 

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType,StringType, DoubleType 

2. Define the Schema

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("Index", IntegerType(), True),
    StructField("Customer Id", StringType(), True),
    StructField("First Name", StringType(), True),
    StructField("Last Name", StringType(), True),
    StructField("Company", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Phone 1", StringType(), True),
    StructField("Phone 2", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("Subscription Date", DateType(), True),
    StructField("Website", StringType(), True)
])


3. Read the CSV File 

In [0]:
df = spark.read.csv("/Volumes/manu/default/csv_file", schema=schema, 
header=True) 

In [0]:
display(df.limit(10))

### 4. Load Multiple CSV Files

Load Multiple CSV Files with diffrent schema 

 Ensure that the schema is consistent across all files. 

In [0]:
file_paths = ["/Volumes/manu/default/csv_file/customers-100.csv", "/Volumes/manu/default/csv_file/leads-100.csv", "/Volumes/manu/default/csv_file/people-100.csv"]

In [0]:
df = spark.read.csv(file_paths, header=True, inferSchema=True)

If any file:

Has extra columns → they will be ignored.

Has missing columns → the missing values will be null.

In [0]:
display(df.limit(10))

Load Multiple CSV Files with same schema

In [0]:
file_paths = ["/Volumes/manu/default/csv_file/customers-100.csv", "/Volumes/manu/default/csv_file/customers-1000.csv"]

In [0]:
df = spark.read.csv(file_paths, header=True, inferSchema=True)

In [0]:
display(df.limit(10))

Interview question:
-  How Does inferSchema Work?
- Ans:-Behind the Scenes: When you use inferSchema, Spark runs a job that scans the CSV file from top to bottom to identify the best-suited data type for each column based on the values it encounters.

Does It Make Sense to Use inferSchema?
- Pros: 
 Useful when the schema of the file keeps changing, as it allows Spark to 
automatically detect the data types.

- Cons: 
- Performance Impact: Spark must scan the entire file, which can take extra 
time, especially for large files. 
- Loss of Control: You lose the ability to explicitly define the schema, which may 
lead to incorrect data types if the data is inconsistent.

### Defining Schema as a String

In [0]:
customerSchema = '''
    Index Integer,
    `Customer Id` String,
    `First Name` String,
    `Last Name` String,
    Company String,
    City String,
    Country String,
    `Phone 1` String,
    `Phone 2` String,
    Email String,
    `Subscription Date` Date,
    Website String
'''



Load the DataFrame with the defined schema

In [0]:
df = spark.read.load("/Volumes/manu/default/csv_file/customers-100.csv", format="csv", schema=customerSchema, header=True)

In [0]:
df.printSchema()

-  **Schema Definition**: Both methods define a schema for the DataFrame,accommodating the dataset's requirements, including handling null values where applicable. 
- **Data Types**: The Joining_Date column is defined as StringType to accommodate potential date format issues or missing values. 
- **Loading the DataFrame**: The spark.read.load method is used to load the CSV file into a DataFrame using the specified schema. 
- **Printing the Schema**: The df.printSchema() function allows you to verify that the DataFrame is structured as intended. 

## PySpark Column Selection & Manipulation: Key Techniques

. Different Methods to Select Columns 
- Using col() function/ column() / string way: 



In [0]:
from pyspark.sql.functions import *

In [0]:
#Using col() function 
df.select(col("First Name")).show(10)

In [0]:
#Using column() function 
df.select(column("City")).show(10)

In [0]:
#Directly using string name 
df.select("Website").show()

2. Selecting Multiple Columns Together 

In [0]:
df2 = df.select("Customer Id", "First Name", col("City"), column("Website"), 
df.Email) 
df2.show(10) 

In [0]:
x%md
3. Listing All Columns in a DataFrame 

In [0]:
df.columns

4.Renaming Columns with alias() 

In [0]:
df.select( 
  col("First Name").alias('EmployeeName'),  # Rename "First Name" to "EmployeeName" 
  col("Last Name").alias('L_name'),  # Rename "Last Name" to L_name" 
  column("Company"),  # Select "Company" 
  #df.Subscription Date  # Select "Subscription Date" cannot be done because it has " " between words
  df["Subscription Date"]
).show(10) 

5. Using selectExpr() for Concise Column Selection

selectExpr() allows you to use SQL expressions directly and rename columns 

In [0]:
df.selectExpr("`First Name` as EmployeeName", "`Last Name` as Last_Name","Email as Mail","Company").show(10) 

**Summary **
-  Use col(), column(), or string names to select columns. 
-  Use expr() and selectExpr() for SQL-like expressions and renaming. 
- Use alias() to rename columns. 
- Get the list of columns using df.columns. 

## PySpark DataFrame Manipulation part 2: Adding, Renaming, and Dropping Columns 

1. Adding New Columns with withColumn()

In [0]:
#Add a constant value column:
newdf = df.withColumn("NewColumn", lit(1))

In [0]:
# Add a column based on an expression: 
newdf = df.withColumn("withinCountry", expr("Country == 'India'"))

In [0]:
# display(newdf)

**summery**
- This function allows adding multiple columns, including calculated ones: 
- Example: 
  - Assign a constant value with lit(). 
  - Perform calculations using existing columns like multiplying values. 

2. Renaming Columns with withColumnRenamed() 

PySpark provides the withColumnRenamed() method to rename columns. This is especially useful when you want to change the names for clarity or to follow naming conventions: 

In [0]:
new_df = df.withColumnRenamed("Email", "mail")

In [0]:
#• Handling column names with special characters or spaces: If a column has specialcharacters or spaces, you need to use backticks (`) to escape it:
newdf.select("`First Name`").show(5) 

3. Dropping Columns with drop()

To remove unwanted columns, you can use the drop() method:

In [0]:
df2 = df.drop("Country")

In [0]:
df2.limit(5).display()

In Spark, DataFrames are immutable by nature. This means that after creating a DataFrame,its contents cannot be changed. All transformations like adding, renaming, or droppingcolumns result in a new DataFrame, keeping the original one intact. 
-  For instance, dropping columns creates a new DataFrame without altering the original:

In [0]:
newdf = df.drop("First Name", "Country")

In [0]:
newdf.limit(5).display()

-- **_Key Points_** 
-   Use withColumn() for adding columns, with lit() for constant values and expressionsfor computed values. 
-  Use withColumnRenamed() to rename columns and backticks for special characters or spaces. 
-   Use drop() to remove one or more columns. 
-   DataFrames are immutable in Spark—transformations result in new DataFrames,leaving the original unchanged

##  changing data types, filtering data, and handling unique/distinct values in PySpark

1.Changing Data Types (Schema Transformation)


In PySpark, you can change the data type of a column using the cast() method. This is helpful when you need to convert data types.

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col
# Change the 'Index' column from integer to double 
df = df.withColumn("Index", col("Index").cast("double")) 

In [0]:
df.printSchema()

In [0]:
#HW
# Convert 'Subscription Date' column to string 
df2 = df.withColumn("Subscription Date", col("Subscription Date").cast("string"))
df2.printSchema() 

2. Filtering Data 

You can filter rows based on specific conditions. For instance, to filter **Index**  greater than 90

In [0]:
filtered_df = df.filter(col("Index") > 90) 
filtered_df.show() 

3. Multiple Filters (Chaining Conditions) 

You can also apply multiple conditions using & or | (AND/OR) to filter data.

In [0]:
filtered_df = df.filter((df["Index"] < 30) & (df["City"] == "Isabelborough"))
filtered_df.show()

4. Filtering on Null or Non-Null Values

Filtering based on whether a column has NULL values or not is crucial for data cleaning

In [0]:
df.display()

# Filter rows where 'Address' is NULL 


In [0]:
# using this code to create null values 
df3 = df.withColumn("City", when(col("Index") > 98, None).otherwise(col("City")))

In [0]:
display(df3)

In [0]:
# Filter rows where 'City' is NULL
filtered_df = df3.filter(df["City"].isNull()) 
filtered_df.show()

In [0]:
## HW
# Filter rows where 'Country' is NOT NULL 
filtered_df = df.filter(df["City"].isNotNull()) 
filtered_df.show(10)

## Handling Unique or Distinct Data 

In [0]:
# Get distinct rows from the entire DataFrame
unq_df  = df.distinct()
unq_df.show()

In [0]:
# Get distinct values from the 'City' column
unique_City_df = df.select("City").distinct() 
unique_City_df.show() 

To remove duplicates based on specific columns, such as Email or Phone, use dropDuplicates(): 

In [0]:
# Remove duplicates based on 'Email' column 
unique_df = df.dropDuplicates(["Email"]) 
unique_df.show()

In [0]:
df.printSchema()

In [0]:
# Remove duplicates based on both 'Phone' and 'Email' 
unique_df = df.dropDuplicates(["Phone 1", "Email"]) 
unique_df.show() 

## 6. Counting Distinct Values

In [0]:
# Count distinct values in the 'City' column 
distinct_count_City = df.select("City").distinct().count() 
print("Distinct City Count:", distinct_count_City)

In [0]:
# # Count distinct combinations of 'Department' and 
# 'Performance_Rating' 
# distinct_combinations_count = df.select("Department", 
# "Performance_Rating").distinct().count() 
# print("Distinct Department and Performance Rating Combinations:", 
# distinct_combinations_count) 